<b><font size="6" color="#E8800A">Week 12b: Neural networks for a numeric target</font></b><br>
<b><font size="4">A price prediction is a forward pass through learned weights</font></b><br>

An MLP learns curved relationships by combining weighted sums with nonlinear activations. Price prediction also exposes a practical constraint: the units of the target change the optimisation problem.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>From a neuron to a price</font>](#neuron)
* [<font color='#E8800A'>The MLP reference on the course data</font>](#reference)
* [<font color='#E8800A'>MLP settings</font>](#mlp_settings)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

In [ ]:
import sys
import json
import joblib
from sklearn.pipeline import Pipeline
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import copy
import time
import warnings
from functools import partial
from joblib import parallel_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, result_context, results_before

random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
set_matplotlib_formats("svg")
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (7, 5)})

ignore_unknown_category_warning = partial(
    warnings.filterwarnings, "ignore",
    message=r"^Found unknown categories in columns \[.*\] during transform\. These unknown categories will be encoded as all zeros$",
    category=UserWarning, module=r"^sklearn\.preprocessing\._encoders$")
ignore_unknown_category_warning()

from preprocessing import (
    FoldPreprocessor,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"


__Step 1:__ Load the cars, set aside the test rows, and prepare the training folds.

In [ ]:
cars, regression_recipe = load_recipe("../../data/interim/cars4you.csv", regression_recipe_log)
X = cars.loc[:, regression_recipe["categorical"] + regression_recipe["numeric"]].copy()
y = cars["price"]
# The test rows are sealed once: 20 percent of the cars, and no choice below reads them.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state)
cars_folds = outer_splitter("regression", random_state=random_state)
# `cars_prepared` holds, for each of the ten folds, the rows to fit (`X_fit`, `y_fit`) and the rows to score (`X_score`, `y_score`), prepared inside the fold. The recipe is also fitted once on all the training rows, so every model fitted on all the rows learns from `X_train_ready` and is scored on `X_test_ready`.
with parallel_config(backend="loky", initializer=ignore_unknown_category_warning):
    cars_prep = FoldPreprocessor(regression_recipe).fit(X_train, y_train)
    X_train_ready = cars_prep.transform(X_train)
    X_test_ready = cars_prep.transform(X_test)
    cars_prepared = prepared_folds(FoldPreprocessor(regression_recipe), cars_folds,
                                   X_train, y_train, n_jobs=4)
print(f"{len(y_train):,} training cars and {len(y_test):,} test cars; {cars_folds.get_n_splits()} validation folds")

# <font color='#E8800A'>From a neuron to a price</font> <a class="anchor" id="neuron"></a>
[Back to TOC](#toc)

A linear regression predicts from one weighted sum. A multilayer perceptron, or **MLP**, makes several weighted sums in a hidden layer, changes them with an activation function, and combines them into a prediction.

For one car with prepared inputs $x_1,\ldots,x_p$, hidden unit $j$ computes
$$z_j=\sum_{i=1}^{p}x_iw_{ij}+b_j,\qquad h_j=\max(0,z_j).$$
Here $p$ is the number of inputs, $w_{ij}$ is the weight from input $i$ to unit $j$, and $b_j$ is that unit's bias. The sum $z_j$ is its input before activation; $h_j$ is its output after **ReLU**, which replaces negative values with zero.

For $m$ hidden units the output is
$$\hat y_{\mathrm{std}}=\sum_{j=1}^{m}h_jv_j+c,\qquad
\hat y=\mu_y+\sigma_y\hat y_{\mathrm{std}}.$$
Here $\hat y_{\mathrm{std}}$ is the standardised predicted price and $\hat y$ is the predicted price in euros. The output weight $v_j$ multiplies hidden value $h_j$ and $c$ is the output bias. The output activation is **identity**, so this sum is left unchanged. The last equation returns the standardised prediction to euros using the training target mean $\mu_y$ and standard deviation $\sigma_y$.

Training adjusts all weights and biases to reduce half squared prediction error, with an L2 penalty on large weights. Raw-target and standardised-target losses use different units and cannot be compared directly. The forward pass predicts; backpropagation calculates how each weight affects the loss, and the optimiser uses those derivatives to update the weights.

<div class="alert alert-block alert-info">

**Low training loss does not guarantee low validation error.** The regression output predicts numeric prices, and ReLU can represent changes in slope. Predictions on unseen cars can be poor even when training loss is small and the stopping rule has been satisfied.

</div>

# <font color='#E8800A'>The MLP reference on the course data</font> <a class="anchor" id="reference"></a>
[Back to TOC](#toc)

The **reference network** is scikit-learn's default `MLPRegressor`: one hidden layer of 100 ReLU units, the Adam optimiser, an L2 penalty `alpha=0.0001` and at most 200 iterations. The target is standardised inside each training fold, and predictions return to euros before MAE is calculated.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html'>sklearn.neural_network.MLPRegressor(hidden_layer_sizes=(100,), activation='relu', solver='adam', alpha=0.0001, max_iter=200, random_state=33)</a>

**Definition:**
A feedforward neural network for numeric predictions, fitted by iterative optimisation.

**Main Parameters:**
- `hidden_layer_sizes`: `(100,)` means one hidden layer with 100 units; `(100, 100)` means two such layers.
- `activation`: the hidden-layer function; `identity`, `logistic`, `tanh` and `relu` change how each weighted sum is passed on.
- `max_iter`: the training budget, measured in passes through the training rows for Adam. Reaching it can leave the fit unfinished.
- `random_state`: fixes initial weights and shuffling; 33 is held fixed throughout.

**Main Attributes (after fitting):**
- `coefs_` and `intercepts_`: learned weight matrices and bias vectors, in layer order.
- `n_iter_`: completed training iterations; `loss_`: final half squared-error plus L2 penalty in the fitted target units, rather than MAE.
- `loss_curve_`: the training objective after each iteration; `out_activation_`: the output activation.

**Main Methods:**
- `fit(X, y)` learns the weights.
- `predict(X)` performs a forward pass.
</div>

__Step 2:__ Score the default network on the ten prepared folds. Use this regressor inside the target transformer:

```python
MLPRegressor(random_state=random_state)
```

In [ ]:
default_network = TransformedTargetRegressor(
    regressor=...,
    transformer=StandardScaler())
train_scores, validation_scores, fold_warnings = [], [], []
for fold in tqdm(cars_prepared, desc='cars prepared', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted_fold = clone(default_network).fit(fold["X_fit"], fold["y_fit"])
    # A fit that reaches the iteration limit raises a ConvergenceWarning; it is shown once here and counted below.
    for item in caught:
        warnings.warn(item.message, item.category)
    fold_warnings.append(len(caught) > 0)
    train_scores.append(mean_absolute_error(fold["y_fit"], fitted_fold.predict(fold["X_fit"])))
    validation_scores.append(mean_absolute_error(fold["y_score"], fitted_fold.predict(fold["X_score"])))
print(f"{sum(fold_warnings)} of {len(fold_warnings)} folds raised a ConvergenceWarning")

__Step 3:__ Fit the same configuration on the prepared training cars.

In [ ]:
fitted_default = clone(default_network).fit(X_train_ready, y_train)
print(f"iterations: {fitted_default.regressor_.n_iter_}, final loss in standardised-target units: {fitted_default.regressor_.loss_:.6f}")

__Step 4:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = fitted_default.regressor_
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = r"\hat y_{\mathrm{std}}"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 5:__ Place the drawn units and name their nodes.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]


__Step 6:__ Extract the fitted weights connecting the drawn units.

In [ ]:
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)


__Step 7:__ Label each layer with its fitted activation and array shapes.

In [ ]:
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + ("\n" + r"$\hat y=\mu_y+\sigma_y\hat y_{\mathrm{std}}$ in euros")])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 8:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The fitted reference network")
net_ax.axis("off")
plt.tight_layout()
plt.show()

**The fitted arrays connect 21 inputs to 100 ReLU units and those units to one identity output.** The drawing shows six inputs and six hidden units, with ellipses marking the omitted units. Blue weights are positive and orange weights negative; the largest absolute network weight is 1.572. The target scaler converts the output back to euros.

__Step 9:__ Open the log with the fixed network's CV and held-out MAE in euros.

In [ ]:
cars_log = []
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": cars_folds.get_n_splits(), "name": "MLP (100,), relu, target standardised (untuned)",
    "train": train_scores, "validation": validation_scores, "model": fitted_default},
    X_test_ready, y_test, metric=mean_absolute_error, label="MAE")
default_row = cars_log[0].copy()

**The default network reaches cross-validated MAE 2,565.90 EUR and test MAE 2,582.21 EUR, with training MAE 2,297.45 EUR.** All ten validation fits raise a ConvergenceWarning. The fit on all training cars also reaches 200 iterations, ending at loss 0.071461 in standardised-target units.

# <font color='#E8800A'>MLP settings</font> <a class="anchor" id="mlp_settings"></a>
[Back to TOC](#toc)

Each comparison uses the same ten training folds. The scaling comparison varies feature and target standardisation; the other sweeps change one constructor argument at a time. The joint network combines the settings with the lowest validation MAE in each comparison.

| Parameter | Meaning |
| --- | --- |
| scaling | Whether the feature scaler is kept and whether the target is standardised inside each training fold. |
| `activation` | Function applied to each hidden weighted sum. |
| `hidden_layer_sizes` | Units in each hidden layer; its length is the number of hidden layers. |
| `max_iter` | Optimisation budget in epochs for Adam; reaching it can leave the fit unfinished. |
| `batch_size` | Rows in each minibatch update; the default `'auto'` is 200. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)

### `MLP: scaling`

The four combinations vary feature scaling and target standardisation separately. Standardising prices changes the size of training errors and gradients. `TransformedTargetRegressor` fits its target scaler inside each training fold and returns predictions to euros.

For inputs without feature scaling, a copy of the recipe sets `scaler="none"`. `FoldPreprocessor` and `prepared_folds` reprocess the raw `X_train` with the same `cars_folds` splitter; they do not inverse-transform the scaled folds. The other transformations and feature selection are fitted inside each training fold, so selected columns may differ. The preparation fitted on all training rows transforms `X_test` for assessment. The candidate list is `["unscaled X, raw y", "inherited X, raw y", "unscaled X, standardised y", "inherited X, standardised y"]`.

__Step 10:__ Prepare the training folds again without the feature scaler, keeping the rest of the recipe.

In [ ]:
with parallel_config(backend="loky", initializer=ignore_unknown_category_warning):
    unscaled_recipe = copy.deepcopy(regression_recipe)
    unscaled_recipe["scaler"] = "none"
    unscaled_prep = FoldPreprocessor(unscaled_recipe).fit(X_train, y_train)
    X_train_unscaled = unscaled_prep.transform(X_train)
    X_test_unscaled = unscaled_prep.transform(X_test)
    unscaled_prepared = prepared_folds(
        FoldPreprocessor(unscaled_recipe), cars_folds, X_train, y_train, n_jobs=4)
print(f"{len(unscaled_prepared)} folds prepared without the feature scaler")

In [ ]:
# The four arms: feature scaler removed or kept, target raw or standardised inside each training fold; each arm names its folds and the rows its all-rows fit uses.
mlp_scaling_arms = [
    ("unscaled X, raw y", unscaled_prepared, X_train_unscaled, X_test_unscaled, False),
    ("inherited X, raw y", cars_prepared, X_train_ready, X_test_ready, False),
    ("unscaled X, standardised y", unscaled_prepared, X_train_unscaled, X_test_unscaled, True),
    ("inherited X, standardised y", cars_prepared, X_train_ready, X_test_ready, True),
]
mlp_scaling_values = [arm[0] for arm in mlp_scaling_arms]
# One row per arm will hold its scores, its seconds and its fitted model.
mlp_scaling_rows = []


__Step 11:__ Score each candidate and keep its fitted model.

In [ ]:
for label, arm_folds, arm_train, arm_test, scale_target in tqdm(mlp_scaling_arms, desc='mlp scaling arms', leave=True, position=0):
    started = time.perf_counter()
    candidate = MLPRegressor(random_state=random_state)
    if scale_target:
        candidate = TransformedTargetRegressor(regressor=candidate, transformer=StandardScaler())
    train_scores_sweep, validation_scores_sweep, fold_warnings = [], [], []
    for fold in arm_folds:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = clone(candidate).fit(fold["X_fit"], fold["y_fit"])
        train_scores_sweep.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores_sweep.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = clone(candidate).fit(arm_train, y_train)
    network = deployed.regressor_ if scale_target else deployed
    mlp_scaling_rows.append({"configuration": label + ", default settings", "setting": label,
        "seconds": time.perf_counter() - started, "train_MAE": np.mean(train_scores_sweep), "cv_MAE": np.mean(validation_scores_sweep),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(arm_test)), "iterations": network.n_iter_,
        "learned values": sum(w.size for w in network.coefs_) + sum(b.size for b in network.intercepts_),
        "fold warnings": sum(fold_warnings), "train": train_scores_sweep, "validation": validation_scores_sweep,
        "parameters": candidate.get_params(deep=False), "model": deployed, "input": arm_test})


__Step 12:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per arm, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_scaling_sweep = pd.DataFrame(mlp_scaling_rows)
mlp_scaling_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "learned values", "fold warnings", "seconds"]].round(2)

In [ ]:
# The sweep's winner joins the log.
mlp_scaling_winner = mlp_scaling_sweep.loc[mlp_scaling_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": cars_folds.get_n_splits(),
    "name": mlp_scaling_winner["configuration"], "train": mlp_scaling_winner["train"],
    "validation": mlp_scaling_winner["validation"], "model": mlp_scaling_winner["model"]},
    mlp_scaling_winner["input"], y_test, "illustrative, untuned", metric=mean_absolute_error, label="MAE")

__Step 13:__ Plot the training and validation MAE for `scaling`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_scaling_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_scaling_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_scaling_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="scaling", xticks=parameter_positions, xticklabels=mlp_scaling_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**With inherited features, standardising the target lowers validation MAE from 11,903.30 to 2,565.90 EUR at the same 200-iteration budget.** Both fits still warn on all ten folds. Without the feature scaler, standardising the target gives 4,602.28 EUR and no fold warnings, so satisfying the stopping rule does not guarantee lower prediction error.

The following curves act on hidden weighted sums. The regression output remains identity; composing identity hidden layers leaves one affine prediction, a weighted sum plus bias.

In [ ]:
activation_input = np.linspace(-4, 4, 401)
activation_values = {"identity": activation_input, "logistic": 1 / (1 + np.exp(-activation_input)),
                     "tanh": np.tanh(activation_input), "relu": np.maximum(0, activation_input)}
activation_formulas = {"identity": r"$h(z)=z$", "logistic": r"$h(z)=1/(1+e^{-z})$",
                       "tanh": r"$h(z)=\tanh(z)$", "relu": r"$h(z)=\max(0,z)$"}
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharex=True)
for ax, (label, values) in zip(axes, activation_values.items()):
    ax.plot(activation_input, values, linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.7)
    ax.axvline(0, color="grey", linewidth=0.7)
    ax.set(title=label + "\n" + activation_formulas[label], xlabel="Weighted sum z", xlim=(-4, 4))
    ax.set_ylim({"identity": (-4.3, 4.3), "logistic": (-0.1, 1.1),
                 "tanh": (-1.1, 1.1), "relu": (-0.3, 4.3)}[label])
axes[0].set_ylabel("Hidden activation h(z)")
fig.suptitle("Hidden activations: each panel has its own output scale")
plt.tight_layout()
plt.show()

### `MLP: activation`

The `activation` is the function applied to each **hidden** weighted sum $z$; it does not affect the output layer, which produces the regression prediction directly (or its standardised form when the target scaler is active). The default is `relu`.

| Activation | Formula | Range |
| --- | --- | --- |
| `identity` | $h(z) = z$ | $(-\infty,\,+\infty)$ |
| `logistic` | $h(z) = \dfrac{1}{1 + e^{-z}}$ | $(0,\,1)$ |
| `tanh` | $h(z) = \tanh(z)$ | $(-1,\,+1)$ |
| `relu` | $h(z) = \max(0,\,z)$ | $[0,\,+\infty)$ |

With `identity` every hidden layer is an affine map, so successive hidden layers collapse to one affine prediction: a weighted sum plus an intercept. This gives the same functional form as linear regression, although the optimiser and regularisation can produce different fitted coefficients. The sweep holds the architecture, seed, target scaling and training budget at the reference. The candidate list is `["identity", "logistic", "tanh", "relu"]`.

__Step 14:__ Set the `activation` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_activation_values = ["identity", "logistic", "tanh", "relu"]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_activation_rows = []


__Step 15:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_activation_values, desc='mlp activation values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__activation=value)
    train_scores_sweep, validation_scores_sweep, fold_warnings = [], [], []
    for fold in cars_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = clone(candidate).fit(fold["X_fit"], fold["y_fit"])
        train_scores_sweep.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores_sweep.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    network = deployed.regressor_
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_activation_rows.append({"configuration": "TransformedTargetRegressor(regressor=MLPRegressor(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + "), transformer=StandardScaler())", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_MAE": np.mean(train_scores_sweep), "cv_MAE": np.mean(validation_scores_sweep),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test_ready)), "iterations": network.n_iter_,
        "learned values": sum(w.size for w in network.coefs_) + sum(b.size for b in network.intercepts_),
        "fold warnings": sum(fold_warnings), "train": train_scores_sweep, "validation": validation_scores_sweep,
        "parameters": candidate.get_params(deep=False), "model": deployed})


__Step 16:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_activation_sweep = pd.DataFrame(mlp_activation_rows)
mlp_activation_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "learned values", "fold warnings", "seconds"]].round(2)

In [ ]:
# The sweep's winner joins the log.
mlp_activation_winner = mlp_activation_sweep.loc[mlp_activation_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": cars_folds.get_n_splits(),
    "name": mlp_activation_winner["configuration"], "train": mlp_activation_winner["train"],
    "validation": mlp_activation_winner["validation"], "model": mlp_activation_winner["model"]},
    X_test_ready, y_test, "illustrative, untuned", metric=mean_absolute_error, label="MAE")

__Step 17:__ Plot the training and validation MAE for `activation`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_activation_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_activation_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_activation_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="activation", xticks=parameter_positions, xticklabels=mlp_activation_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**ReLU has the lowest validation MAE, 2,565.90 EUR, followed by tanh at 2,714.13, logistic at 2,981.66 and identity at 3,075.69 EUR.** Identity stops after 91 iterations with no fold warnings; ReLU warns on all ten folds. The sweep therefore keeps the reference activation despite its unfinished optimisation.

### `MLP: hidden_layer_sizes`

`hidden_layer_sizes` lists the number of units in each hidden layer. For example, `(100,)` specifies one hidden layer of 100 units.

| `hidden_layer_sizes` | Units in each hidden layer |
| --- | --- |
| `(64, 64, 64)` | 64 units in the first, second and third hidden layers. |
| `(32, 64, 32)` | 32 units in the first hidden layer, 64 in the second and 32 in the third. |
| `(64, 32)` | 64 units in the first hidden layer and 32 in the second. |
| `(32, 64)` | 32 units in the first hidden layer and 64 in the second. |

Each hidden unit computes a weighted sum of the preceding layer's outputs, adds its bias and applies the activation function. Changing layer sizes changes the weight matrices and biases. A closer fit to training rows may not improve predictions on validation rows. The candidate list is `[(8,), (32,), (64,), (100,), (32, 32), (32, 32, 32), (64, 64, 64), (32, 64, 32), (64, 32), (32, 64)]`.

__Step 18:__ Set the `hidden_layer_sizes` values.

In [ ]:
mlp_hidden_layer_sizes_values = ...

In [ ]:
# The values of this one setting were set above; every other setting stays at the reference.
# One row per value will hold its scores, its seconds and its fitted model.
mlp_hidden_layer_sizes_rows = []


__Step 19:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_hidden_layer_sizes_values, desc='mlp hidden layer sizes values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__hidden_layer_sizes=value)
    train_scores_sweep, validation_scores_sweep, fold_warnings = [], [], []
    for fold in cars_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = clone(candidate).fit(fold["X_fit"], fold["y_fit"])
        train_scores_sweep.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores_sweep.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    network = deployed.regressor_
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_hidden_layer_sizes_rows.append({"configuration": "TransformedTargetRegressor(regressor=MLPRegressor(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + "), transformer=StandardScaler())", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_MAE": np.mean(train_scores_sweep), "cv_MAE": np.mean(validation_scores_sweep),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test_ready)), "iterations": network.n_iter_,
        "learned values": sum(w.size for w in network.coefs_) + sum(b.size for b in network.intercepts_),
        "fold warnings": sum(fold_warnings), "train": train_scores_sweep, "validation": validation_scores_sweep,
        "parameters": candidate.get_params(deep=False), "model": deployed})


__Step 20:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_hidden_layer_sizes_sweep = pd.DataFrame(mlp_hidden_layer_sizes_rows)
mlp_hidden_layer_sizes_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "learned values", "fold warnings", "seconds"]].round(2)

In [ ]:
# The sweep's winner joins the log.
mlp_hidden_layer_sizes_winner = mlp_hidden_layer_sizes_sweep.loc[mlp_hidden_layer_sizes_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": cars_folds.get_n_splits(),
    "name": mlp_hidden_layer_sizes_winner["configuration"], "train": mlp_hidden_layer_sizes_winner["train"],
    "validation": mlp_hidden_layer_sizes_winner["validation"], "model": mlp_hidden_layer_sizes_winner["model"]},
    X_test_ready, y_test, "illustrative, untuned", metric=mean_absolute_error, label="MAE")

__Step 21:__ Plot the training and validation MAE for `hidden_layer_sizes`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_hidden_layer_sizes_sweep))
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(parameter_positions, mlp_hidden_layer_sizes_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_hidden_layer_sizes_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="hidden_layer_sizes", xticks=parameter_positions, xticklabels=mlp_hidden_layer_sizes_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**The 100-unit reference has the lowest validation MAE, 2,565.90 EUR.** The `(64, 64, 64)`, `(32, 64, 32)`, `(64, 32)` and `(32, 64)` networks reach validation MAE 2,600.60, 2,577.60, 2,573.28 and 2,612.48 EUR. The equal-width three-layer network lowers training MAE to 2,145.05 EUR and raises no fold warning, but its validation error remains above the reference. The extra layers improve the fit to training cars without improving validation here.

### `MLP: max_iter`

`max_iter` is the optimisation budget: with Adam, one iteration is one pass through the training rows. The reference allows 200. A ConvergenceWarning means the budget ended before the stopping rule was satisfied; no warning does not establish good predictions. The completed iteration count and fold warnings describe when optimisation stopped. The lowest validation MAE selects the iteration budget for the joint network; test MAE does not affect this choice. The candidate list is `[20, 200, 1000, 2000]`.

__Step 22:__ Set the `max_iter` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_max_iter_values = [20, 200, 1000, 2000]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_max_iter_rows = []


__Step 23:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_max_iter_values, desc='mlp max iter values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__max_iter=value)
    train_scores_sweep, validation_scores_sweep, fold_warnings = [], [], []
    for fold in cars_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = clone(candidate).fit(fold["X_fit"], fold["y_fit"])
        train_scores_sweep.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores_sweep.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    network = deployed.regressor_
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_max_iter_rows.append({"configuration": "TransformedTargetRegressor(regressor=MLPRegressor(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + "), transformer=StandardScaler())", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_MAE": np.mean(train_scores_sweep), "cv_MAE": np.mean(validation_scores_sweep),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test_ready)), "iterations": network.n_iter_,
        "learned values": sum(w.size for w in network.coefs_) + sum(b.size for b in network.intercepts_),
        "fold warnings": sum(fold_warnings), "train": train_scores_sweep, "validation": validation_scores_sweep,
        "parameters": candidate.get_params(deep=False), "model": deployed})


__Step 24:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_max_iter_sweep = pd.DataFrame(mlp_max_iter_rows)
mlp_max_iter_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "learned values", "fold warnings", "seconds"]].round(2)

In [ ]:
# The sweep's winner joins the log.
mlp_max_iter_winner = mlp_max_iter_sweep.loc[mlp_max_iter_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": cars_folds.get_n_splits(),
    "name": mlp_max_iter_winner["configuration"], "train": mlp_max_iter_winner["train"],
    "validation": mlp_max_iter_winner["validation"], "model": mlp_max_iter_winner["model"]},
    X_test_ready, y_test, "illustrative, untuned", metric=mean_absolute_error, label="MAE")

__Step 25:__ Plot the training and validation MAE for `max_iter`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_max_iter_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_max_iter_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_max_iter_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="max_iter", xticks=parameter_positions, xticklabels=mlp_max_iter_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**The 200-iteration budget has the lowest validation MAE, 2,565.90 EUR, against 3,191.91 at 20 and 2,596.54 at both larger budgets.** The full-training fit stops after 249 iterations with either a 1,000- or 2,000-iteration allowance, and neither raises a fold warning; all ten folds warn at 20 and 200. Training MAE falls from 2,297.45 at 200 to 2,271.07 EUR at the larger budgets while validation error rises. The joint network therefore keeps 200. Test MAE falls from 2,582.21 to 2,527.91 EUR at the larger budgets, but it is not used to select the budget.

### `MLP: batch_size`

Adam updates the weights on one minibatch of rows at a time, and one pass through all the training rows is an epoch, so `batch_size` sets how many updates an epoch holds: 64 rows give many small updates and 512 rows give fewer, smoother ones. The default `'auto'` is a batch of 200 rows, so the reference is the 200 entry, and each `max_iter` iteration is one epoch. The candidate list is `[64, 200, 512]`.

__Step 26:__ Set the `batch_size` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_batch_size_values = [64, 200, 512]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_batch_size_rows = []


__Step 27:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_batch_size_values, desc='mlp batch size values', leave=True, position=0):
    started = time.perf_counter()
    candidate = clone(default_network).set_params(regressor__batch_size=value)
    train_scores_sweep, validation_scores_sweep, fold_warnings = [], [], []
    for fold in cars_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = clone(candidate).fit(fold["X_fit"], fold["y_fit"])
        train_scores_sweep.append(mean_absolute_error(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores_sweep.append(mean_absolute_error(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = clone(candidate).fit(X_train_ready, y_train)
    network = deployed.regressor_
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in candidate.regressor.get_params().items() if item != MLPRegressor().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_batch_size_rows.append({"configuration": "TransformedTargetRegressor(regressor=MLPRegressor(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + "), transformer=StandardScaler())", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_MAE": np.mean(train_scores_sweep), "cv_MAE": np.mean(validation_scores_sweep),
        "test_MAE": mean_absolute_error(y_test, deployed.predict(X_test_ready)), "iterations": network.n_iter_,
        "learned values": sum(w.size for w in network.coefs_) + sum(b.size for b in network.intercepts_),
        "fold warnings": sum(fold_warnings), "train": train_scores_sweep, "validation": validation_scores_sweep,
        "parameters": candidate.get_params(deep=False), "model": deployed})


__Step 28:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores in euros, the test score, the fit diagnostics and the seconds spent.
mlp_batch_size_sweep = pd.DataFrame(mlp_batch_size_rows)
mlp_batch_size_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "iterations", "learned values", "fold warnings", "seconds"]].round(2)

In [ ]:
# The sweep's winner joins the log.
mlp_batch_size_winner = mlp_batch_size_sweep.loc[mlp_batch_size_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": cars_folds.get_n_splits(),
    "name": mlp_batch_size_winner["configuration"], "train": mlp_batch_size_winner["train"],
    "validation": mlp_batch_size_winner["validation"], "model": mlp_batch_size_winner["model"]},
    X_test_ready, y_test, "illustrative, untuned", metric=mean_absolute_error, label="MAE")

__Step 29:__ Plot the training and validation MAE for `batch_size`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_batch_size_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_batch_size_sweep["train_MAE"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_batch_size_sweep["cv_MAE"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="batch_size", xticks=parameter_positions, xticklabels=mlp_batch_size_sweep["setting"], ylabel="MAE (EUR)")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Batch size 200 has the lowest validation MAE, 2,565.90 EUR, against 2,596.61 at 64 and 2,618.05 EUR at 512.** The smallest training fold has 2,560 cars, so the default 'auto' uses 200 rows and reproduces the reference score exactly. Size 64 lowers training MAE to 2,252.15 EUR while its validation error rises.

__Step 30:__ Confirm that the default `'auto'` batch is the 200 entry.

In [ ]:
# 'auto' means min(200, rows). Every fit here has more than 200 rows, so the batch_size=200 row repeats the reference fit.
smallest_fit = min(len(fold["y_fit"]) for fold in cars_prepared)
print(f"rows in the smallest fit: {smallest_fit}, so 'auto' is a batch of {min(200, smallest_fit)}")
print("batch_size=200 scores the same validation MAE as the reference:", mlp_batch_size_sweep.set_index("setting").loc["200", "cv_MAE"] == default_row["validation MAE"])

__Step 31:__ Refit each batch size on the first prepared fold, stopped after k iterations, and read its training loss and its validation score.

In [ ]:
# Refit every batch size on the first prepared fold, stopped after k iterations; the validation rows of that fold score each refit and no test row is read.
curve_fold = cars_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 3000]
curve_points = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    for curve_value in tqdm(mlp_batch_size_values, desc='mlp batch size values', leave=True, position=0):
        curve_rows = []
        for curve_k in curve_budgets:
            # A fit stopped after k iterations is the k-th iterate of the full fit, because the seed is fixed.
            curve_fit = clone(default_network).set_params(**{"regressor__batch_size": curve_value, "regressor__max_iter": curve_k}).fit(curve_fold["X_fit"], curve_fold["y_fit"])
            curve_rows.append([curve_fit.regressor_.n_iter_, curve_fit.regressor_.loss_, mean_absolute_error(curve_fold["y_score"], curve_fit.predict(curve_fold["X_score"]))])
            if curve_fit.regressor_.n_iter_ < curve_k:
                break
        curve_points[str(curve_value)] = np.array(curve_rows)
for curve_value, curve_rows in curve_points.items():
    print(f"batch size {curve_value}: loss {curve_rows[-1, 1]:.4f} and validation MAE {curve_rows[-1, 2]:,.2f} after {int(curve_rows[-1, 0])} iterations")

__Step 32:__ Tabulate those readings.

In [ ]:
# One row per batch size and stopping point: the iterations reached, the training loss and the validation MAE.
curve_table = pd.DataFrame([{"batch size": curve_value, "iterations": int(curve_row[0]), "training loss": curve_row[1], "validation MAE": curve_row[2]}
                            for curve_value, curve_rows in curve_points.items() for curve_row in curve_rows])
display(curve_table.round(4))

__Step 33:__ Plot the training loss and the validation score of each batch size against the iteration.

In [ ]:
# One panel with two axes: the training loss is solid and read on the left, the validation MAE is dashed and read on the right.
curve_fig, curve_ax = plt.subplots(figsize=(9, 5.2))
curve_ax2 = curve_ax.twinx()
for (curve_value, curve_rows), (curve_colour, curve_marker) in zip(curve_points.items(), [("#33A367", "o"), ("#0D7A3E", "s"), ("#053D20", "^")]):
    curve_ax.plot(curve_rows[:, 0], curve_rows[:, 1], color=curve_colour, marker=curve_marker, ms=5, lw=2.2, label=f"batch size {curve_value}")
    curve_ax2.plot(curve_rows[:, 0], curve_rows[:, 2], color=curve_colour, marker=curve_marker, ms=5, lw=2.2, linestyle="--")
curve_ax.set_xscale("log")
curve_ax.set_ylim(bottom=0)
curve_ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
curve_ax.grid(axis="x", visible=False)
curve_ax2.grid(False)
curve_ax.legend(loc="upper right")
curve_ax.text(0.5, -0.2, "solid: training loss (left axis); dashed: validation MAE (right axis)", transform=curve_ax.transAxes, ha="center", va="top", fontsize=11)
curve_ax.set(xlabel="iteration (log scale)", ylabel="training loss\n(half squared error + L2 penalty,\nstandardised target)", title="Adam with three batch sizes")
curve_ax2.set_ylabel("validation MAE, fold 1 (EUR)")
plt.tight_layout()
curve_fig.subplots_adjust(bottom=0.24)
plt.show()

**On fold 1, batch sizes 64, 200 and 512 finish at validation MAE 2,742.06, 2,733.51 and 2,700.88 EUR after 151, 222 and 413 iterations.** Their training losses are 0.0623, 0.0631 and 0.0627 in standardised-target units. Each curve refits that fold's training cars and scores its validation cars, so these endpoints describe one fold rather than the ten-fold sweep.

__Step 34:__ Combine the best value of every sweep, read from the sweep tables, into the joint network.

### The joint network

The joint network uses the activation, hidden layer sizes, iteration budget and batch size with the lowest validation MAE in their respective tables. The scaling comparison selects its feature scaler and target standardisation. The reference value is available in every sweep.

In [ ]:
# The best validation value of each sweep, read from its table.
joint_sweeps = {"activation": mlp_activation_sweep, "hidden_layer_sizes": mlp_hidden_layer_sizes_sweep, "max_iter": mlp_max_iter_sweep, "batch_size": mlp_batch_size_sweep}
joint_best = {setting: table.loc[table["cv_MAE"].idxmin()] for setting, table in joint_sweeps.items()}
joint_settings = {setting: row["parameters"]["regressor"].get_params()[setting] for setting, row in joint_best.items()}
# The scaling sweep names the best arm: its folds, its all-rows preparation, and whether the target is standardised.
joint_arm_label, joint_folds, joint_train, joint_test, joint_scale_target = mlp_scaling_arms[mlp_scaling_sweep["cv_MAE"].idxmin()]
joint_prep = unscaled_prep if joint_folds is unscaled_prepared else cars_prep
joint_regressor = MLPRegressor(random_state=random_state, **joint_settings)
joint_network = TransformedTargetRegressor(regressor=joint_regressor, transformer=StandardScaler()) if joint_scale_target else joint_regressor
reference_settings = default_network.regressor.get_params()
joint_choices = pd.DataFrame({
    "reference": [reference_settings[setting] for setting in joint_sweeps] + ["inherited X, standardised y"],
    "joint value": [joint_settings[setting] for setting in joint_sweeps] + [joint_arm_label],
    "validation MAE": [row["cv_MAE"] for row in joint_best.values()] + [mlp_scaling_sweep["cv_MAE"].min()]},
    index=[*joint_sweeps, "scaling"])
joint_choices.round(2)

__Step 35:__ Score the joint network on the ten prepared folds, as every sweep row was scored.

In [ ]:
joint_train_scores, joint_validation_scores, joint_fold_warnings = [], [], []
for fold in tqdm(joint_folds, desc='joint folds', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted_fold = clone(joint_network).fit(fold["X_fit"], fold["y_fit"])
    joint_train_scores.append(mean_absolute_error(fold["y_fit"], fitted_fold.predict(fold["X_fit"])))
    joint_validation_scores.append(mean_absolute_error(fold["y_score"], fitted_fold.predict(fold["X_score"])))
    joint_fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
print(f"{sum(joint_fold_warnings)} of {len(joint_fold_warnings)} folds raised a ConvergenceWarning")

__Step 36:__ Fit the joint network on the prepared training cars and log it.

In [ ]:
joint_fitted = clone(joint_network).fit(joint_train, y_train)
joint_mlp = joint_fitted.regressor_ if joint_scale_target else joint_fitted
joint_name = (f"MLP joint of the sweeps: {joint_settings['hidden_layer_sizes']} {joint_settings['activation']}, "
              f"batch_size={joint_settings['batch_size']}, {joint_arm_label}")
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train), "folds": cars_folds.get_n_splits(),
    "name": joint_name, "train": joint_train_scores, "validation": joint_validation_scores, "model": joint_fitted},
    joint_test, y_test, "chosen on these folds", metric=mean_absolute_error, label="MAE")
joint_row = cars_log[-1].copy()
print(f"iterations: {joint_mlp.n_iter_}, final loss {joint_mlp.loss_:.6f}")

**The joint network keeps ReLU, one 100-unit hidden layer, batch size 200 and inherited feature scaling with a standardised target.** It reproduces the default's validation MAE 2,565.90 EUR and test MAE 2,582.21 EUR, because each joint-selection sweep retains the reference value. Its cross-validated error is optimistic: the folds that chose each value also score their combination. Week 13's nested cross-validation measures a search on outer validation rows that did not choose its settings.

__Step 37:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = joint_mlp
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = r"\hat y_{\mathrm{std}}" if joint_scale_target else r"\hat y"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 38:__ Place the drawn units and name their nodes.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]


__Step 39:__ Extract the fitted weights connecting the drawn units.

In [ ]:
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)


__Step 40:__ Label each layer with its fitted activation and array shapes.

In [ ]:
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + ("\n" + r"$\hat y=\mu_y+\sigma_y\hat y_{\mathrm{std}}$ in euros" if joint_scale_target else "")])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 41:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The joint network")
net_ax.axis("off")
plt.tight_layout()
plt.show()

**The joint network has 21 inputs, 100 ReLU units and one identity output, with largest absolute weight 1.572.** Its fitted arrays match the reference network because the combined sweep winners keep its settings. The identity output is returned to euros by the target scaler.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The board row is the joint network. Its cross-validated MAE averages the ten training validation folds; its test MAE scores the separately held-out cars. Earlier model scores are read from the saved course board, and the no-model floor predicts the training mean for every test car.

__Step 42:__ Read the recorded training, cross-validation and test scores together.

In [ ]:
comparison = pd.DataFrame(cars_log).rename(columns={"validation MAE": "cv MAE"})
display(comparison[["model", "train MAE", "cv MAE", "test MAE"]].round(2))

__Step 43:__ Build the course board with the joint network and the saved earlier results.

In [ ]:
results_dir = "../../results"
context = result_context("../../data/interim/cars4you.csv", regression_recipe_log,
    X_train.index, X_test.index, outer_folds=outer_folds(cars_folds, X_train, y_train))
floor = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
board = Scoreboard("Cars4You", "MAE (EUR)", floor=floor, context=context)
for earlier in results_before(12, "regression", results_dir, context=context):
    board.record(*earlier)
board.record(12, joint_row["model"], joint_row["validation MAE"], joint_row["test MAE"])
print(board.table())
board.save(results_dir, 12, "regression",
           baseline_cv=default_row["validation MAE"], baseline_test=default_row["test MAE"])

# Package the already fitted preparation and joint network without fitting them again.
chosen_model = Pipeline([("preparation", joint_prep), ("model", joint_fitted)])
joblib.dump(chosen_model, f"{results_dir}/week_12_regression_model.joblib")
# The joint network's settings, for the week that starts from it.
joint_file = Path(results_dir) / "mlp" / "joint_regression.json"
joint_file.parent.mkdir(parents=True, exist_ok=True)
joint_file.write_text(json.dumps({"model": joint_row["model"], "metric": "MAE (EUR)", "cv": joint_row["validation MAE"], "test": joint_row["test MAE"],
                                  "settings": {**joint_settings, "hidden_layer_sizes": list(joint_settings["hidden_layer_sizes"])},
                                  "scaling": joint_arm_label, "target": "standardised" if joint_scale_target else "raw",
                                  "fitted_model": "../week_12_regression_model.joblib"}, indent=1) + "\n", encoding="utf-8")

**The joint network enters at validation MAE 2,565.90 EUR and test MAE 2,582.21 EUR, against the fixed-mean floor of 6,843.46 EUR.** The saved voting ensemble has lower errors, 2,208.81 and 2,192.92 EUR. The saved KNN has higher validation error, 2,880.73 EUR, but lower test error, 2,564.17 EUR, so those two estimates rank it differently from the network.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. A price is predicted by a forward pass: weighted sums, ReLU, an identity output and the inverse of the target standardisation.
2. Target units affect iterative learning, so standardising the target is part of the network rather than a detail of the data.
3. Scaling, activation, depth and width, and batch size change different parts of the calculation. The max_iter sweep compares the optimisation budget, completed iterations and warnings with prediction error.
4. The joint network combines the best value of every sweep and is the board row. Its cross-validated error is optimistic, because the folds that chose each value also score the combination; Week 13's nested cross-validation measures a search on outer validation rows.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

Explain the two transformations and the validation estimate in words.

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- [Scikit-learn: MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).
- [Scikit-learn: neural network models](https://scikit-learn.org/stable/modules/neural_networks_supervised.html).
- [Scikit-learn: TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html).